In [ ]:
!pip install qdrant-client sentence-transformers

In [ ]:
!pip install -q "qdrant-client[fastembed]"

In [ ]:
from google.colab import userdata
key = userdata.get("QDRANT_API_KEY")
print("Key loaded:", bool(key), "| length:", len(key) if key else 0)

Key loaded: True | length: 176


In [ ]:
from qdrant_client import QdrantClient
from google.colab import userdata

client = QdrantClient(
    url="https://735fd0e9-0ba8-4a2c-ab2c-d41e5bdb5640.eu-central-1-0.aws.cloud.qdrant.io:6333",
    api_key=userdata.get("QDRANT_API_KEY"),
)

print(client.get_collections())   # sanity check — should print collections=[]

collections=[CollectionDescription(name='aa'), CollectionDescription(name='article'), CollectionDescription(name='articles'), CollectionDescription(name='my_collection')]


# Create the collection

In [ ]:
from qdrant_client import models

client.create_collection(
    collection_name="Article",
    vectors_config=models.VectorParams(
        size=384,                          # matches all-MiniLM-L6-v2
        distance=models.Distance.COSINE,
    ),
)

True

In [ ]:
print(client.get_collections())   # should now list 'articles'

collections=[CollectionDescription(name='Article'), CollectionDescription(name='aa'), CollectionDescription(name='article'), CollectionDescription(name='articles'), CollectionDescription(name='my_collection')]


# Load embedding model and ingest data

In [ ]:
from qdrant_client import models

documents = [
    {"id": 1, "text": "Car repair guide", "category": "automotive"},
    {"id": 2, "text": "How to cook pasta", "category": "food"},
]

points = [
    models.PointStruct(
        id=doc["id"],
        vector=models.Document(text=doc["text"], model="sentence-transformers/all-MiniLM-L6-v2"),
        payload={"title": doc["text"], "category": doc["category"]},
    )
    for doc in documents
]

client.upload_points(collection_name="Article", points=points)
print("Ingested", len(points), "points")

Ingested 2 points


# Create payload index

In [ ]:
client.create_payload_index(
    collection_name="Article",
    field_name="category",
    field_schema="keyword",
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

# Query with a filter

In [ ]:
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue

results = client.query_points(
    collection_name="Article",  # lowercase, matches what you created
    query=models.Document(
        text="automobile maintenance",
        model="sentence-transformers/all-MiniLM-L6-v2",  # same model as ingestion
    ),
    query_filter=Filter(
        must=[FieldCondition(key="category", match=MatchValue(value="automotive"))]
    ),
    limit=3,
)

for r in results.points:
    print(f"Score: {r.score:.3f}  |  {r.payload['title']}")

Score: 0.590  |  Car repair guide
